# Chapter 12 Lab: Compressed Sensing

This notebook accompanies **Chapter 12**, the final chapter. It builds
the L1-minimization, basis pursuit (LP), coherence, and RIP-constant
machinery, then works through all five lab exercises -- including a
`cvxpy`-based noisy recovery experiment in the last one.

Run the setup cell first. If `cvxpy` is not installed, run
`!pip install cvxpy` in a new cell before Lab 12.5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog, minimize_scalar
from scipy.linalg import null_space
from itertools import combinations
import random

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def canonical_dual(F):
    return np.linalg.solve(F @ F.T, F)

def dual_family_direction(F):
    return null_space(F)

def basis_pursuit(A, y):
    '''
    Solve min ||c||_1 s.t. Ac = y via linear programming.
    Reformulation: min 1^T t  s.t.  Ac = y,  -t <= c <= t,  t >= 0.
    '''
    n, m = A.shape
    c_obj = np.zeros(2 * m); c_obj[m:] = 1.0
    A_eq = np.hstack([A, np.zeros((n, m))])
    b_eq = y
    A_ub = np.block([[ np.eye(m), -np.eye(m)],
                     [-np.eye(m), -np.eye(m)]])
    b_ub = np.zeros(2 * m)
    bounds = [(None, None)] * m + [(0, None)] * m
    res = linprog(c_obj, A_ub=A_ub, b_ub=b_ub,
                  A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    return res.x[:m]

def coherence(A):
    G = A.T @ A
    off = np.abs(G - np.diag(np.diag(G)))
    return off.max()

def welch_bound(n, m):
    return np.sqrt((m - n) / (n * (m - 1)))

def rip_constant(A, s, max_subsets=2000):
    n, m = A.shape
    all_subsets = list(combinations(range(m), s))
    if len(all_subsets) > max_subsets:
        all_subsets = random.sample(all_subsets, max_subsets)
    delta = 0.0
    for T in all_subsets:
        A_T = A[:, list(T)]
        eigs = np.linalg.eigvalsh(A_T.T @ A_T)
        delta = max(delta, abs(eigs[-1] - 1), abs(1 - eigs[0]))
    return delta

f1 = np.array([1.0, 0.0])
f2 = np.array([-0.5, np.sqrt(3)/2])
f3 = np.array([-0.5, -np.sqrt(3)/2])
F_tri = np.column_stack([f1, f2, f3])

## Lab Exercise 12.1: Reproducing Example 12.1

(a) Verify $t^*=1/\sqrt3$ gives $c^*=(1,0,0)^T$ for $x=f_1$. (b) For $x=\alpha f_2$, verify $c^*=(0,\alpha,0)^T$. (c) For generic $x=(2,-1)^T$, verify exactly two nonzero entries.

In [ ]:
S = F_tri @ F_tri.T
k = dual_family_direction(F_tri)[:, 0]

def l1_minimize_over_family(x):
    c_canonical = F_tri.T @ np.linalg.solve(S, x)
    def l1_obj(t):
        return np.sum(np.abs(c_canonical + t * k))
    result = minimize_scalar(l1_obj, bounds=(-5, 5), method='bounded')
    return c_canonical + result.x * k, result.x

c_star_a, t_star_a = None, None  # TODO: l1_minimize_over_family(f1)
print(f"Part (a): t*={t_star_a:.6f} (expect 1/sqrt(3)={1/np.sqrt(3):.6f})")
print(f"          c*={c_star_a.round(6)} (expect (1,0,0))")

alpha = 2.5
x_b = alpha * f2
c_star_b, t_star_b = None, None  # TODO: l1_minimize_over_family(x_b)
print(f"\nPart (b): c*={c_star_b.round(6)} (expect (0,{alpha},0))")

x_c = np.array([2.0, -1.0])
c_star_c, t_star_c = None, None  # TODO: l1_minimize_over_family(x_c)
n_nonzero = np.sum(np.abs(c_star_c) > 1e-6)
print(f"\nPart (c): c*={c_star_c.round(6)}, nonzero entries={n_nonzero} (expect 2)")

## Lab Exercise 12.2: Basis Pursuit via LP

(a) Triangular frame, $x=f_1$: verify $c^*=(1,0,0)^T$. (b) Random $20\times100$ matrix, 3-sparse signal: exact recovery? (c) Sweep sparsity $s=1,\ldots,10$; does the coherence bound predict failure?

In [ ]:
y_a = F_tri @ np.array([1.0, 0.0, 0.0])
c_bp_a = None  # TODO: basis_pursuit(F_tri, y_a)
print(f"Part (a): c* = {c_bp_a.round(6)} (expect (1,0,0))")

rng = np.random.default_rng(42)
n_meas, m_sig = 20, 100
A = rng.standard_normal((n_meas, m_sig))
A = A / np.linalg.norm(A, axis=0)

x_true = np.zeros(m_sig)
support_true = rng.choice(m_sig, size=3, replace=False)
x_true[support_true] = rng.standard_normal(3) * 2
y_b = A @ x_true

x_rec_b = None  # TODO: basis_pursuit(A, y_b)
err_b = np.linalg.norm(x_rec_b - x_true)
print(f"\nPart (b): recovery error = {err_b:.2e}, exact recovery: {err_b < 1e-6}")

print("\n=== Part (c): sparsity sweep ===")
mu_A = coherence(A)
coherence_threshold = 0.5 * (1 + 1/mu_A)
print(f"Coherence mu = {mu_A:.4f}, coherence bound predicts exact recovery for s < {coherence_threshold:.2f}")

for s in range(1, 11):
    x_s = np.zeros(m_sig)
    support_s = rng.choice(m_sig, size=s, replace=False)
    x_s[support_s] = rng.standard_normal(s) * 2
    y_s = A @ x_s
    x_rec_s = basis_pursuit(A, y_s)
    err_s = np.linalg.norm(x_rec_s - x_s)
    success = err_s < 1e-6
    print(f"s={s:>2}: error={err_s:.2e}, success={success}")

*Your answer: does the coherence bound correctly predict the failure threshold observed in part (c)?* (replace this text)

## Lab Exercise 12.3: The Coherence-RIP Connection

For random Gaussian $n=20,m=80$: (a) compute $\mu$, $\delta_2$. (b) verify $\delta_2\le\mu$. (c) repeat for $s=3,4,5$. (d) is the bound tight or loose?

In [ ]:
rng2 = np.random.default_rng(7)
n2, m2 = 20, 80
A2 = rng2.standard_normal((n2, m2))
A2 = A2 / np.linalg.norm(A2, axis=0)

mu_2 = coherence(A2)
print(f"Coherence mu = {mu_2:.4f}")

for s in [2, 3, 4, 5]:
    delta_s = None  # TODO: rip_constant(A2, s)
    bound = (s - 1) * mu_2
    print(f"s={s}: delta_s={delta_s:.4f}, (s-1)*mu={bound:.4f}, "
          f"bound holds={delta_s <= bound + 1e-9}, tightness ratio={delta_s/bound:.3f}")

*Your answer: is the coherence-RIP bound tight or loose here? What does this suggest about the relative sharpness of the two guarantee types?* (replace this text)

## Lab Exercise 12.4: ETFs as Sensing Matrices

Compare the triangular frame to (a) random Gaussian and (b) $[[1,0,1],[0,1,0]]$ normalized. Compute $\mu$, sparsity bound, $\delta_2$ for each. Is the ETF best?

In [ ]:
rng3 = np.random.default_rng(3)
A_random = rng3.standard_normal((2, 3))
A_random = A_random / np.linalg.norm(A_random, axis=0)

A_struct = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0]])
A_struct = A_struct / np.linalg.norm(A_struct, axis=0)

candidates = [
    ("Triangular (ETF)", F_tri),
    ("Random Gaussian", A_random),
    ("Structured [[1,0,1],[0,1,0]]", A_struct),
]

print(f"{'Matrix':<32} {'mu':>8} {'s-bound':>9} {'delta_2':>9}")
for name, A in candidates:
    mu = None       # TODO: coherence(A)
    s_bound = None  # TODO: 0.5*(1 + 1/mu)
    delta2 = None   # TODO: rip_constant(A, 2)
    print(f"{name:<32} {mu:>8.4f} {s_bound:>9.4f} {delta2:>9.4f}")

wb = welch_bound(2, 3)
print(f"\nWelch bound for n=2, m=3: {wb:.4f}")

*Your ranking and explanation: is the triangular frame (ETF) the best sensing matrix here, and why (connect to the Welch bound)?* (replace this text)

## Lab Exercise 12.5 (Challenge): Noisy Measurements

(a) Implement noisy basis pursuit via `cvxpy`. (b) For $20\times80$ sensing matrix, 2-sparse signal, sweep $\sigma=0.01,0.05,0.1,0.2$. (c) verify roughly linear error scaling.

**Note:** this exercise requires `cvxpy`. If not installed, run `!pip install cvxpy` first.

In [ ]:
import cvxpy as cp

def noisy_basis_pursuit(A, y, eta):
    '''Solve min ||z||_1 s.t. ||Az - y||_2 <= eta using cvxpy.'''
    m = A.shape[1]
    z = cp.Variable(m)
    objective = None    # TODO: cp.Minimize(cp.norm(z, 1))
    constraints = None  # TODO: [cp.norm(A @ z - y, 2) <= eta]
    problem = cp.Problem(objective, constraints)
    problem.solve()
    return z.value

rng4 = np.random.default_rng(11)
n4, m4 = 20, 80
A4 = rng4.standard_normal((n4, m4))
A4 = A4 / np.linalg.norm(A4, axis=0)

x_true4 = np.zeros(m4)
support4 = rng4.choice(m4, size=2, replace=False)
x_true4[support4] = rng4.standard_normal(2) * 2

sigmas = [0.01, 0.05, 0.1, 0.2]
recovery_errors = []

for sigma in sigmas:
    noise = rng4.normal(0, sigma, size=n4)
    y_noisy = A4 @ x_true4 + noise
    eta = 2 * sigma * np.sqrt(n4)

    x_rec4 = None  # TODO: noisy_basis_pursuit(A4, y_noisy, eta)
    err4 = np.linalg.norm(x_rec4 - x_true4)
    recovery_errors.append(err4)
    print(f"sigma={sigma}: recovery error = {err4:.4f}")

plt.figure(figsize=(7,5))
plt.plot(sigmas, recovery_errors, 'o-')
plt.xlabel('sigma (noise std)')
plt.ylabel('||x_hat - x||')
plt.title('Recovery error vs. noise level')
plt.show()

ratios = np.array(recovery_errors) / np.array(sigmas)
print(f"\nerror/sigma ratios: {ratios.round(3)} (roughly constant => linear scaling)")

*Your observation: does the recovery error scale roughly linearly with sigma, as predicted?* (replace this text)